<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week1/week1_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# EYE OF RA — Standard Environment Header
# Paste this as the FIRST cell in every new notebook
# Run it before anything else
# ============================================================

# Install libraries not pre-loaded in Colab
# (runs fast after first time — Colab caches these)
!pip install -q pdfplumber shap xgboost streamlit plotly \
    openpyxl folium streamlit-folium camelot-py

# ============================================================
# Core imports — used across the entire project
# ============================================================

# Data handling
import pandas as pd
import numpy as np

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

# Explainability
import shap

# PDF extraction
import pdfplumber

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# System
import os
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================
print("=" * 50)
print("✅ Eye of Ra environment ready")
print("All systems go. Let's catch some corruption. 🇬🇭")
print("=" * 50)

✅ Eye of Ra environment ready
All systems go. Let's catch some corruption. 🇬🇭


In [9]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Step 1: Verify all our PDF files are here and readable
# ============================================================

import os
import pdfplumber

# List all PDF files we uploaded
pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"✅ Found {len(pdf_files)} PDF files\n")
print("=" * 50)
for f in pdf_files:
    print(f"  📄 {f}")
print("=" * 50)

✅ Found 14 PDF files

  📄 ppa_annual_report_2024.pdf
  📄 ppa_bulletin_2021_nov_dec.pdf
  📄 ppa_bulletin_2022_jul_aug.pdf
  📄 ppa_bulletin_2022_mar_apr.pdf
  📄 ppa_bulletin_2022_nov_dec.pdf
  📄 ppa_bulletin_2023_jul_aug.pdf
  📄 ppa_bulletin_2023_mar_apr.pdf
  📄 ppa_bulletin_2023_may_jun.pdf
  📄 ppa_bulletin_2023_nov_dec.pdf
  📄 ppa_bulletin_2023_sep_oct.pdf
  📄 ppa_bulletin_2024_jan_feb.pdf
  📄 ppa_bulletin_2024_mar_apr.pdf
  📄 ppa_bulletin_2024_may_jun.pdf
  📄 ppa_bulletin_2024_nov_dec.pdf


In [12]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 2: Verify all PDF files are uploaded and readable
# (with error handling for corrupted uploads)
# ============================================================

pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"Found {len(pdf_files)} PDF files. Checking each one...\n")
print("=" * 50)

good_files = []
bad_files = []

for filename in pdf_files:
    filepath = f'/content/{filename}'
    try:
        with pdfplumber.open(filepath) as pdf:
            num_pages = len(pdf.pages)
            first_page_text = pdf.pages[0].extract_text()

            if first_page_text:
                preview = first_page_text[:100].replace('\n', ' ')
                print(f"✅ {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   Preview: {preview}...")
                good_files.append(filename)
            else:
                print(f"⚠️  {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   WARNING: No readable text on page 1")
                bad_files.append(filename)
    except Exception as e:
        print(f"❌ {filename}")
        print(f"   ERROR: {str(e)[:80]}")
        bad_files.append(filename)
    print()

print("=" * 50)
print(f"✅ Readable files: {len(good_files)}")
print(f"❌ Problem files:  {len(bad_files)}")

if bad_files:
    print("\nThese files need to be re-uploaded:")
    for f in bad_files:
        print(f"  → {f}")

print("=" * 50)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 2 - PDF verification with error handling"
# ------------------------------------------------------------

Found 14 PDF files. Checking each one...

✅ ppa_annual_report_2024.pdf
   Pages: 86
   Preview: EXECUTIVE SUMMARY Introduction This Annual Report has been prepared in fulfillment of sections 3 (i)...

✅ ppa_bulletin_2021_nov_dec.pdf
   Pages: 16
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2021 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_jul_aug.pdf
   Pages: 12
   Preview: Public Procurement Authority: Electronic Bulletin Jul-Aug 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_mar_apr.pdf
   Pages: 11
   Preview: Public Procurement Authority: Electronic Bulletin Mar-Apr 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2022_nov_dec.pdf
   Pages: 14
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2022 Submit 2021 Procurement Plan Using PP...

✅ ppa_bulletin_2023_jul_aug.pdf
   Pages: 23
   Preview: Public Procurement Authority: Electronic Bulletin Jul-Aug 2023 Submit 2023 Procurement Plan Us

In [13]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 3: Peek inside one bulletin — understand the structure
# ============================================================

# We'll use the most recent bulletin as our reference
sample_file = '/content/ppa_bulletin_2024_nov_dec.pdf'

print("DEEP INSPECTION: ppa_bulletin_2024_nov_dec.pdf")
print("=" * 60)

with pdfplumber.open(sample_file) as pdf:

    # Look at pages 2, 3, and 4 — page 1 is usually a cover
    for page_num in [1, 2, 3]:
        page = pdf.pages[page_num]
        print(f"\n{'=' * 60}")
        print(f"PAGE {page_num + 1}")
        print(f"{'=' * 60}")

        # Check 1: Is there plain text?
        text = page.extract_text()
        if text:
            print("\n--- RAW TEXT (first 500 characters) ---")
            print(text[:500])
        else:
            print("\n⚠️ No plain text found on this page")

        # Check 2: Are there tables?
        tables = page.extract_tables()
        if tables:
            print(f"\n--- TABLES FOUND: {len(tables)} table(s) on this page ---")
            for i, table in enumerate(tables):
                print(f"\nTable {i+1} — {len(table)} rows x {len(table[0])} columns")
                print("First 3 rows:")
                for row in table[:3]:
                    print(f"  {row}")
        else:
            print("\n⚠️ No tables found on this page")

print("\n" + "=" * 60)
print("✅ Structure inspection complete")
print("=" * 60)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 3 - bulletin structure inspection"
# ------------------------------------------------------------

DEEP INSPECTION: ppa_bulletin_2024_nov_dec.pdf

PAGE 2

--- RAW TEXT (first 500 characters) ---
Public Procurement Authority: Electronic Bulletin Nov-Dec 2024
e-Bulletin
PPA LAUNCHES THE METHODOLOGY FOR ASSESSING
In this Edition
PROCUREMENT SYSTEMS (MAPS) ASSESSMENT FOR GHANA
 PPA launches the
Methodology for
Assessing
Procurement
Systems (MAPS)
Assessment for
Ghana -Pg. 2,11
 Online
Procurement
Submissions–
Pg. 3, 4,5 & 6
 Editorial - Pg. 8,9
 End of Year
Message from CEO
– Pg.10
On November 26, 2024, the Public Procurement Authority (PPA) in collaboration with the
 Photo Story -
Min

⚠️ No tables found on this page

PAGE 3

--- RAW TEXT (first 500 characters) ---
Public Procurement Authority: Electronic Bulletin Nov-Dec 2024
2024 PROCUREMENT PLAN SUBMISSIONS ON GHANEPS AS AT NOVEMBER 2024
1. Abetifi Presbyterian College Of Education 62. Berekum East Municipal Assembly
2. Ablekuma Central Municipal Assembly 63. Bia East District Assembly
3. Ablekuma North Municipal Assembly 64. B

In [14]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 4: Scan ALL pages to find contract award data
# ============================================================

# Keywords that signal contract award data
CONTRACT_KEYWORDS = [
    'contract', 'award', 'tender', 'supplier', 'amount',
    'GH₵', 'GHS', 'cedis', 'bid', 'procur', 'lot'
]

sample_file = '/content/ppa_bulletin_2024_nov_dec.pdf'

print("SCANNING ALL PAGES FOR CONTRACT DATA")
print("=" * 60)

interesting_pages = []

with pdfplumber.open(sample_file) as pdf:
    total_pages = len(pdf.pages)
    print(f"Total pages: {total_pages}\n")

    for page_num in range(total_pages):
        page = pdf.pages[page_num]
        text = page.extract_text()
        tables = page.extract_tables()

        if not text:
            continue

        # Check if this page contains contract-related keywords
        text_lower = text.lower()
        matches = [kw for kw in CONTRACT_KEYWORDS if kw in text_lower]

        if matches:
            interesting_pages.append(page_num)
            print(f"📋 PAGE {page_num + 1} — Keywords found: {matches}")

            # Show first 300 characters of this page
            preview = text[:300].replace('\n', ' ')
            print(f"   Preview: {preview}...")

            # Report tables if any
            if tables:
                print(f"   Tables: {len(tables)} found")
                for i, table in enumerate(tables):
                    if table and table[0]:
                        cols = len(table[0])
                        rows = len(table)
                        print(f"   Table {i+1}: {rows} rows x {cols} cols")
                        print(f"   Headers: {table[0]}")
            print()

print("=" * 60)
print(f"✅ Found {len(interesting_pages)} pages with contract-related content")
print(f"   Pages: {[p+1 for p in interesting_pages]}")
print("=" * 60)

# ----------------------Add cell 4 - full page scan for contract data--------------------------------------
# COMMIT MESSAGE: ""
# ------------------------------------------------------------

SCANNING ALL PAGES FOR CONTRACT DATA
Total pages: 19

📋 PAGE 1 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 Submit 2024 Procurement Plan Using PPA’s Online Procurement Planning System (http://planning.ppaghana.org/) Page 1...

📋 PAGE 2 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 e-Bulletin PPA LAUNCHES THE METHODOLOGY FOR ASSESSING In this Edition PROCUREMENT SYSTEMS (MAPS) ASSESSMENT FOR GHANA  PPA launches the Methodology for Assessing Procurement Systems (MAPS) Assessment for Ghana -Pg. 2,11  Online Procure...

📋 PAGE 3 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 2024 PROCUREMENT PLAN SUBMISSIONS ON GHANEPS AS AT NOVEMBER 2024 1. Abetifi Presbyterian College Of Education 62. Berekum East Municipal Assembly 2. Ablekuma Central Municipal Assembly 63. Bia East District Assembly 3. Ablekuma North Mun...
   Ta